# 第3课：Python 数据结构——高效组织数据的艺术

> **学习目标**：掌握 list、tuple、dict、set 四种核心数据结构的原理与选择

---

## 为什么需要数据结构？

30 个学生要存名字，你不会写 30 个变量：

```python
student1 = "小明"  # 写到 student30？太荒唐了
```

**数据结构 = 把散落的数据组织成有结构的整体**。就像冰箱分区：蔬菜冷藏、肉冷冻、饮料门架——有组织才能高效。

### 先从内存模型说起

Python 中所有变量都是**指向对象的引用**（指针）。当你写 `x = [1, 2, 3]` 时：

```
内存布局示意：

    栈 (Stack)             堆 (Heap)
    ┌────────┐          ┌──────────────┐
    │   x    │──────────>│ 列表 [1,2,3] │
    └────────┘          └──────────────┘
    （存地址）            （存真实数据）
```

- **栈（Stack）**：变量名 x 存储在栈上，里面保存的是堆内存的地址（引用）
- **堆（Heap）**：列表对象 [1, 2, 3] 实际存储在堆上，由 Python 的垃圾回收器管理
- **变量是标签**：`b = a` 只是复制引用，两个标签指向同一个堆对象——这是后面"复制陷阱"的根源

选择哪种数据结构，本质就是选择不同的内存布局和算法策略，直接影响程序的性能和正确性。

### 四种核心结构一瞥

| 结构 | 内存模型 | 类比 | 核心特点 |
|------|---------|------|---------|
| list | 动态数组（连续指针数组） | 超市购物车 | 有序、可改、可重复 |
| tuple | 固定数组（连续不可变） | 石碑碑文 | 有序、不可改 |
| dict | 哈希表（散列桶+探测链） | 新华字典 | 键值映射、O(1)查找 |
| set | 哈希表（只有键无值） | 花名册 | 无序、不重复、O(1)检查 |

| 操作 | list | tuple | dict | set |
|------|------|-------|------|-----|
| 按索引/键访问 | O(1) | O(1) | O(1)* | - |
| 按值查找 | O(n) | O(n) | O(1)* | O(1)* |
| 末尾添加 | O(1)** | - | O(1)* | O(1)* |
| 成员检查 (in) | O(n) | O(n) | O(1)* | O(1)* |

> *O(1) 是平均情况，哈希冲突严重时可能退化为 O(n)；**均摊 O(1)，扩容时会有一次性 O(n) 开销


---

## 2. 列表 (List)——动态数组的优雅封装

**就像超市购物车**：可放任何东西，按顺序排列，随时增删替换。但 Python 的列表远比购物车强大——它是基于**动态数组**实现的高性能数据结构。

### 内存模型：连续指针数组

```
底层结构 (PyListObject):
+-------------------------------------------+
|  ob_item:  PyObject**  ----> 指针数组      |
|  allocated: 8 (总容量)                     |
|  size: 5 (已用个数)                        |
+-------------------------------------------+

指针数组（连续内存）：
+------+------+------+------+------+------+------+------+
| obj0 | obj1 | obj2 | obj3 | obj4 |      |      |      |
+--|----+--|----+--|----+--|----+--|----+------+------+---+
   v       v      v      v      v
  a       b      c      d      e          <- 指向实际的 Python 对象
```

- **动态数组**：底层是一块**连续内存**，每个槽位存储一个 PyObject* 指针（8 字节）
- **容量 vs 长度**：allocated 是总容量，size 是已用个数。当 size == allocated 时触发扩容
- **扩容策略**：Python 3.12+ 按约 1.125 倍扩容，复制旧数组到新内存——均摊到每次 append 仍是 O(1)
- **连续内存最关键**：通过 `base + i * 8` 直接跳到第 i 个元素——这就是索引 O(1) 的奥秘

### 时间复杂度速查

| 操作 | O |
|------|---|
| 按索引访问 lst[i] | O(1) —— 指针偏移直接定位 |
| 末尾追加 .append() | O(1)* —— 均摊常数时间 |
| 末尾弹出 .pop() | O(1) —— 只减 size，不移数据 |
| 任意位置插入 .insert(i, v) | O(n) —— 后续元素全部后移 |
| 任意位置删除 .pop(i) | O(n) —— 后续元素全部前移 |
| 按值查找 x in lst | O(n) —— 必须逐个比较 |
| 切片 lst[a:b] | O(k) —— 复制 k 个指针到新数组 |
| 排序 .sort() | O(n log n) —— Timsort 算法 |

### 什么时候用列表？什么时候避免？

**适合用 list：**
- 需要按索引快速访问元素
- 数据按插入顺序存储，主要在末尾增删
- 需要保存重复值
- 需要频繁修改（增删改）元素

**避免用 list：**
- 需要频繁在中间位置插入/删除 -> 用 collections.deque（双向队列）在两端操作更优
- 需要快速按值查找 -> 改用 set 或 dict（O(1) vs O(n)）
- 数据量极大且只读 -> 用 tuple（更省内存）


In [ ]:
# 创建一个空列表，内存中分配了一个空的 list 对象（底层为 PyListObject，初始容量为 0）
empty = []               # 空列表
# 使用字面量创建包含 5 个整数的列表，底层预分配了能容纳这些元素的连续内存数组
nums = [1, 2, 3, 4, 5]
# Python 列表是异构容器，每个元素是指向 Python 对象的指针（8 字节），因此可混合不同类型
mix = [1, "hello", True]   # 可混合类型
# print 调用每个对象的 __repr__() 方法获取字符串表示，列表的 __repr__ 递归处理每个元素
print("空:", empty, "数字:", nums, "混合:", mix)

### 索引为什么从 0 开始？

索引本质是"偏移量"：`起始地址 + i * 元素大小（指针宽度 8 字节）`。从 0 开始，第一个元素就在起始地址上——省一次减法运算。这是继承自 C 语言的设计，贯穿整个计算机体系。

```
内存地址： 0x100    0x108    0x110    0x118    0x120
          +--------+--------+--------+--------+--------+
          | lst[0] | lst[1] | lst[2] | lst[3] | lst[4] |
          +--------+--------+--------+--------+--------+
地址公式： base + 0*8  base + 1*8  base + 2*8  ...
```

**负索引**：Python 自动将 lst[-i] 转换为 lst[len(lst) - i]，本质还是正索引偏移。

### 切片的核心机制

**切片** [start:end:step] **左闭右开**——包含 start，不包含 end。好处：
- `s[:3]` 取前 3 个，`s[3:]` 取第 3 个之后
- 相邻切片无缝衔接：`s[:3]` + `s[3:6]` = 完整

**重要：切片总是创建新列表！** 不会像 NumPy 那样返回视图。

```python
a = [1, 2, 3, 4, 5]
b = a[1:3]        # b = [2, 3]，这是**新列表**，修改 b 不影响 a
```

时间复杂度 O(k)（k = 切片长度），空间复杂度也是 O(k)。对大列表切片会占用大量内存——如果只想遍历，用 itertools.islice() 代替。

### 常见切片陷阱

- **lst[:] = 浅拷贝**：等价于 lst.copy()，复制所有元素引用
- **lst[::-1] 反转**：创建整个列表的逆序副本，O(n) 时间 + O(n) 空间
- **步长不为 1 时**：lst[::2] 每隔一个取一个，底层仍然遍历整个列表
- **切片赋值**：lst[1:3] = [a, b, c] 可以替换切片位置的内容，甚至改变长度


In [ ]:
fruits = ["苹果", "香蕉", "橘子", "葡萄", "西瓜"]
# 正索引从 0 开始，fruits[0] 取第一个元素，时间复杂度 O(1)——通过指针偏移直接计算内存地址
print("fruits[0]:", fruits[0])          # 苹果
# 负索引从 -1 开始，Python 自动将其转换为 len + i（即 -1 转换为 len-1），再执行 O(1) 访问
print("fruits[-1]:", fruits[-1])        # 最后一个：西瓜
# 切片 [start:end] 是左闭右开区间：包含索引 start，不包含索引 end
# 切片返回**新列表**（浅拷贝），时间复杂度 O(k) 其中 k = end - start
print("fruits[1:4]:", fruits[1:4])      # ['香蕉','橘子','葡萄']
# [::-1] 以步长 -1 从头到尾反向切片，等效于反转列表，会创建新列表，O(n) 时间 + O(n) 空间
print("fruits[::-1]:", fruits[::-1])    # 反转
# [::2] 从开头到结尾，步长为 2，每隔一个元素取一个，也是 O(n) 创建新列表
print("fruits[::2]:", fruits[::2])      # 隔一取一

### 修改——列表是可变的 (mutable)

和字符串（不可变）不同，列表可以**原地修改**：通过索引赋值或调用方法时，底层数组的指针被替换为新对象的引用，内存地址不变。

### 可变 vs 不可变：为什么重要？

```python
# 可变：列表修改后，所有引用都看到变化
a = [1, 2, 3]
b = a          # b 引用同一个对象
a.append(4)    # a 和 b 都变成 [1, 2, 3, 4]

# 不可变：字符串"修改"实际上是创建新对象
s = "hello"
t = s
s = s + "!"    # s 指向新字符串 "hello!"，t 仍然指向 "hello"
```

| 类型 | 是否可变 | 背后的原因 |
|------|---------|-----------|
| list, dict, set | 可变 | 需要频繁增删改，原地操作效率高 |
| int, float, str, tuple | 不可变 | 保证哈希值不变（可用于字典键），线程安全 |

### 常见的可变性陷阱

**陷阱 1：修改列表时遍历**

```python
# 错误！在遍历列表时删除元素会导致跳过后续元素
lst = [1, 2, 3, 4, 5]
for x in lst:
    if x % 2 == 0:
        lst.remove(x)   # 删除后索引错位！
print(lst)  # 结果不是期望的
```
正确做法：用列表推导式创建新列表，或遍历副本 lst[:]。

**陷阱 2：可变对象作为函数默认参数**

```python
def add_student(name, cache=[]):   # 默认列表在函数定义时只创建一次！
    cache.append(name)
    return cache

print(add_student("小明"))  # ["小明"]
print(add_student("小红"))  # ["小明", "小红"]  <- 缓存一直在累积！
```
正确做法：`def add_student(name, cache=None): cache = cache or []`

**陷阱 3：列表嵌套修改**

```python
matrix = [[0] * 3] * 3   # 外部 *3 复制的是同一内层列表的引用
matrix[0][0] = 1
print(matrix)  # [[1,0,0], [1,0,0], [1,0,0]] 全部被改！
```
正确做法：`matrix = [[0] * 3 for _ in range(3)]`


In [ ]:
f = ["苹果", "香蕉", "橘子"]
# 通过索引直接赋值修改元素——列表是可变类型，内存中该槽位的指针被替换为新对象的引用，O(1)
f[1] = "草莓"          # 修改
# .append() 在列表末尾追加元素：如果底层数组有剩余空间则直接写入，否则触发扩容
# 扩容策略：约 1.125 倍（Python 3.12+），均摊时间复杂度 O(1)
f.append("葡萄")       # 追加
# .pop() 移除并返回最后一个元素，时间复杂度 O(1)；.pop(i) 指定索引则后续元素前移，为 O(n)
popped = f.pop()       # 弹出最后一个
# .insert(index, value) 在指定位置插入：index 位置及之后的所有元素需要后移一个位置，O(n)
f.insert(1, "蓝莓")    # 插入
print("最终:", f)
print("弹出:", popped)

### 排序：sort 与 sorted

**list.sort()** — 原地修改列表，返回 None。**sorted(list)** — 返回新列表，原列表不变。

key 参数指定排序依据，对每个元素计算一次并缓存（Schwartzian 变换，装饰-排序-去装饰）：

### 排序算法：Timsort

Python 使用 **Timsort** 算法——一种融合归并排序和插入排序的混合算法：

- **最佳情况 O(n)**：数据已经基本有序时，只需一次扫描
- **平均/最坏 O(n log n)**：随机数据表现稳定
- **自适应**：自动检测数据中的"自然有序片段"（run），合并时利用这些片段
- **稳定排序**：相同 key 的元素保持原始相对顺序——多级排序时很关键

### 排序最佳实践

| 需求 | 写法 |
|------|------|
| 原地排序 | lst.sort() |
| 保留原列表 | new_lst = sorted(lst) |
| 按长度排序 | sorted(words, key=len) |
| 降序排列 | sorted(lst, reverse=True) 或 lst.sort(reverse=True) |
| 按多个条件 | sorted(lst, key=lambda x: (x.age, x.name)) |
| 自定义比较 | functools.cmp_to_key(old_cmp_func) |

### key 参数为什么比 cmp 快？

key 函数对每个元素**只调用一次**，结果缓存在并行数组中。排序时比较的是缓存值，而不是反复调用比较函数。对于复杂对象或昂贵计算，差距可能是数量级的。

```python
# 方式 1：用 key（推荐，每个元素只调用 len() 一次）
words.sort(key=len)

# 方式 2：用 cmp（已废弃，不推荐）
from functools import cmp_to_key
words.sort(key=cmp_to_key(lambda a, b: len(a) - len(b)))
```


In [ ]:
nums = [3, 1, 4, 1, 5]
# list.sort() 是**原地排序**（in-place），直接修改原列表，不创建新列表，返回 None
# 使用 Timsort 算法（融合归并排序与插入排序），最坏/平均 O(n log n)
# 原地排序的优势：节省内存，不需要额外 O(n) 空间存放结果
nums.sort()                    # 原地
print("sort:", nums)

nums2 = [3, 1, 4]
# sorted() 是**内置函数**，返回一个**新排序后的列表**，原列表保持不变
# 同样使用 Timsort 算法，但多一次 O(n) 的列表复制开销
# 适合需要保留原始数据的场景，或者排序任何可迭代对象（sorted 接受任意 iterable）
print("sorted:", sorted(nums2), "原:", nums2)

# key 参数：sort/sorted 都支持，指定一个"键函数"决定排序依据
# 关键优化：key 函数对每个元素只调用一次，结果缓存在一个并行数组中，避免重复计算
# 这与 "Schwartzian transform"（装饰-排序-去装饰）原理相同
words = ["python", "java", "c", "javascript"]
# 按字符串长度排序（从短到长），len 作为 key 函数
words.sort(key=len)
print("按长度:", words)

# lambda 表达式创建匿名函数，这里 x["s"] 取出每个字典中 "s" 字段的值作为排序键
# 更高效且比定义具名函数更简洁；如果 key 逻辑复杂还是建议用 def
students = [{"n": "小明", "s": 85}, {"n": "小红", "s": 92}]
students.sort(key=lambda x: x["s"])
print("按分数:", students)

---

## 3. 列表的"复制陷阱"

`b = a` 不是复制，只是两个标签指向**同一个对象**。修改 b 会改 a。

```
内存中：
                    +-----------+
              a --->|  [1,2,3]  |<--- b
                    +-----------+
            a 和 b 是同一个对象的两个名字，不是两个列表！
```

### 为什么 Python 用引用语义？

Python 中"万物皆对象"，变量存储的是对象的引用（指针）。这与 C/Java 不同：

- **Python 风格**：`b = a` 意味着 a 和 b 共享同一个对象。优点是赋值操作极快（只复制 8 字节指针），缺点是容易意外修改
- **对比 C++**：传递对象时会调用拷贝构造函数，创建独立副本（安全但昂贵）
- **何时需要副本**：当你希望修改不影响原列表时，必须显式创建副本

### 深浅拷贝的本质区别

```
浅拷贝（shallow copy）:
原列表                 新列表
+----------+          +----------+
| PyObj* 0 |--> str --| PyObj* 0 |  <- 共享内层对象
| PyObj* 1 |--> list -| PyObj* 1 |  <- 内层列表共享！
+----------+          +----------+

深拷贝（deepcopy）:
原列表                 新列表
+----------+          +----------+
| PyObj* 0 |--> str   | PyObj* 0 |--> str（新副本）
| PyObj* 1 |--> list  | PyObj* 1 |--> list（新副本）
+----------+          +----------+
                       全部独立！
```

- **浅拷贝适用**：一维列表、元素全是不可变类型（int、str 等）
- **深拷贝适用**：嵌套的可变对象（列表的列表、字典的列表等），需要完全隔离


In [ ]:
a = [1, 2, 3]
# 严重警告：b = a 只是让 b**引用**同一个列表对象，并没有复制！
# id(b) == id(a)，a 和 b 两个变量名指向同一块内存。修改其中任何一个，另一个也会看到变化
b = a                    # 只是贴标签
b.append(4)
# b 添加元素后，a 也跟着变了——因为 a 和 b 是同一个对象的两个名字
print("a:", a)           # [1, 2, 3, 4]

# .copy() 方法创建**浅拷贝**（shallow copy），生成一个全新的列表对象
# 等价写法：a[:] 或 list(a)，都是 O(n) 时间复杂度，复制所有元素的引用到新数组
c = a.copy()
c.append(5)
# c 是独立的新列表，修改 c 不会影响原列表 a
print("a:", a, "c:", c)  # a 没变

# 浅拷贝的局限：只复制最外层列表，内层嵌套的可变对象仍然共享引用
x = [[1, 2], [3, 4]]
y = x.copy()
# y[0] 和 x[0] 指向同一个 [1, 2] 列表对象——因为浅拷贝只复制外层指针
y[0].append(99)
# x[0] 也被修改了！说明浅拷贝后内层列表仍是共享的
print("x[0]:", x[0])     # [1, 2, 99] — 内层被改了！

from copy import deepcopy
# deepcopy（深拷贝）递归复制所有嵌套对象，构建完全独立的对象树
# 代价：更慢、更占内存；内部使用 memo 字典处理循环引用和重复引用
z = deepcopy(x)
z[0].append(100)
# deepcopy 后的 z 完全独立，嵌套列表也被递归复制，修改 z 不影响 x
print("x[0]:", x[0])

---

## 4. 列表推导式——Pythonic 精髓

一行完成"遍历 -> 过滤 -> 转换 -> 收集"：

```python
[表达式 for 变量 in 可迭代对象 if 条件]
```

### 为什么列表推导式比 for 循环快？

列表推导式底层直接用 **LIST_APPEND 字节码**操作，避免了：
1. 每次循环调用 .append() 方法的属性查找开销
2. 方法调用的栈帧创建开销
3. 中间变量的反复赋值

实测通常比手动 for + append **快 1.5~2 倍**（CPython）。

### 推导式的变体

| 语法 | 结果类型 | 说明 |
|------|---------|------|
| [x for x in ...] | list | 列表推导式 |
| {x for x in ...} | set | 集合推导式，自动去重 |
| {k: v for k, v in ...} | dict | 字典推导式 |
| (x for x in ...) | generator | **生成器表达式**——惰性求值，不立即创建列表 |

### 什么时候**不要**用推导式？

- **逻辑过于复杂**：超过 2 个 for 或 if 时，可读性急剧下降
- **有副作用**：在表达式中做 print() 或修改外部变量——推导式应当只做转换
- **数据极大且只需遍历一次**：用生成器表达式 (x for x in big_data) 节省内存
- **调试不方便**：for 循环中可以加断点，推导式不行

### 嵌套推导式的阅读技巧

```python
# 等价于两重 for 循环
result = [f"{i}x{j}={i*j}" for i in range(1,4) for j in range(1,4)]

# 相当于：
result = []
for i in range(1, 4):
    for j in range(1, 4):
        result.append(f"{i}x{j}={i*j}")
```

阅读时从左到右展开，外层 for 在前，内层 for 在后，最左边表达式是最终结果。


In [ ]:
nums = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# 传统 for 循环方式：手动创建空列表，然后在循环中逐个 .append()
# 效率较低的原因：每次 append 都可能触发底层数组扩容（重新分配+复制），且涉及属性查找和方法调用
squares = []
for n in nums:
    squares.append(n**2)
print("传统:", squares)

# 列表推导式：一行完成"遍历→转换→收集"，是 Pythonic 的核心理念之一
# 底层直接用 LIST_APPEND 字节码操作，避免了属性查找和函数调用开销
# 性能通常比手动 for+append 快 1.5~2 倍（CPython 实测）
# 语法：[表达式 for 变量 in 可迭代对象 if 条件]
squares2 = [n**2 for n in nums]
print("推导:", squares2)

# 带 if 过滤条件的推导式：先遍历 nums，只对偶数的元素计算 n**2 并加入结果列表
# 注意：if 在 for 之后，先过滤再映射（也可理解为 filter + map 的语法糖）
evens = [n**2 for n in nums if n % 2 == 0]
print("偶数平方:", evens)

# 嵌套循环的推导式：外层 for i 先遍历，内层 for j 后遍历
# 等效于两层 for 循环嵌套，执行顺序与普通嵌套 for 循环完全一致
# 注意阅读顺序：从左到右展开，类似普通嵌套循环的书写顺序
table = [f"{i}x{j}={i*j}" for i in range(1,4) for j in range(1,4)]
print("乘法表:", table)

# 展平二维列表：先遍历外层 row，再遍历内层元素 x
# 不要搞反顺序——for 子句按嵌套深度从左到右写，最左边的表达式是最终结果
matrix = [[1,2,3],[4,5,6]]
flat = [x for row in matrix for x in row]
print("展平:", flat)

---

## 5. 元组 (Tuple)——不可变序列

**一旦创建不能修改**。为什么需要不可变？

1. **安全**：防止意外修改——函数返回元组，调用方无法篡改
2. **可哈希**：可作为字典的键（列表不行），因为哈希值不会变化
3. **性能**：创建和访问都比列表略快（下面详解）
4. **语义**：告诉读者"这个数据是固定不变的"

| 场景 | 用... |
|------|-------|
| 需修改 | list |
| 固定配置参数 | tuple |
| 函数多返回值 | tuple（Python 自动打包） |
| 字典键 | tuple（列表不可哈希） |
| 具名字段 | namedtuple（tuple + 字段名） |

### 元组的内存模型：为什么更省内存？

```
内存布局对比：

列表（预留额外容量，对象 + 数组两块内存）：
+------+------+------+------+------+------+------+
| obj0 | obj1 | obj2 |      |      |      |      |  <- 有额外空位
+------+------+------+------+------+------+------+
 allocated = 7, size = 3

元组（单块内存精确分配）：
+------+------+------+
| obj0 | obj1 | obj2 |  <- 不多不少
+------+------+------+
 fixed size = 3
```

- 元组创建时**精确分配**所需内存，不预留额外空间 -> 节省内存
- 元组本身是**单一内存块**（列表是对象 + 数组两块内存）-> 减少内存碎片
- 对于短元组（长度 <= 20），CPython 会**缓存复用**已创建的空元组对象

### 元组的"不可变"到底是什么意思？

元组的**不可变**是指引用本身不可变，而不是引用指向的对象不可变：

```python
t = ([1, 2], "hello")
t[0].append(3)   # 可以！t[0] 是列表，列表本身可变
print(t)         # ([1, 2, 3], "hello") <- 元组内容变了！

t[1] = "world"   # TypeError: 元组元素不能重新赋值
```

- 顶层引用不可改：不能增、删、替换元素
- 内层对象仍可变：元组的"不可变"是**浅层**的（shallow immutability）

### namedtuple：有名字的元组

```python
from collections import namedtuple
Point = namedtuple("Point", ["x", "y"])
p = Point(10, 20)
print(p.x, p.y)      # 10 20 -- 按名字访问，比 p[0] 更清晰
print(p[0])          # 10 -- 同时支持索引访问
```

namedtuple 既有元组的轻量和不可变特性，又有关键字访问的可读性——是定义简单数据容器的优秀选择。


In [ ]:
# 用圆括号创建元组——不可变序列，一旦创建就不能增、删、改
# 底层为 PyTupleObject，固定大小，不支持 resize，因此比列表更节省内存（无需预留额外容量）
t1 = (1, 2, 3)
# 元组的本质由**逗号**决定，而非括号！括号只是用于分组和明确边界
# 这种省略括号的写法叫"元组打包"（tuple packing），Python 自动将逗号分隔的值打包为元组
t2 = 1, 2, 3               # 括号可省（逗号是关键）
print("t1:", t1, type(t1))

# 单元素元组——逗号绝对不能省略！
# (5) 不带逗号只是整数 5 外面包了一层无意义的括号，是普通表达式
single = (5,)               # 元组
not_t = (5)                 # 只是整数
# 推荐：创建单元素元组时始终加上逗号，这是 Python 社区约定
print("single:", single, type(single))
print("not_t:", not_t, type(not_t))

# 元组支持索引和切片（作为序列类型），但不支持赋值修改（TypeError）
# 索引 O(1)，切片 O(k) 创建新元组
t = (10, 20, 30)
print("t[0]:", t[0], "t[:2]:", t[:2])

# 元组解包（tuple unpacking）：将元组中每个元素按位置依次赋值给左侧变量
# 变量数必须与元组长度完全一致，否则 ValueError: too many/few values to unpack
name, age, score = ("小明", 18, 95.5)
print(f"{name}, {age}岁, {score}分")

# Pythonic 变量交换——无需临时变量
# 执行过程：先计算右侧表达式 b, a，构造元组 (20, 10)
# 再将元组解包赋值给左侧 a, b，一步完成交换
a, b = 10, 20
a, b = b, a
# 对比其他语言：temp = a; a = b; b = temp —— Python 更简洁且底层效率更高（使用 ROT_TWO 字节码）
print(f"交换: a={a}, b={b}")

---

## 6. 字典 (Dict)——最强查找表

像新华字典：按"拼音（键）"直接翻到"释义（值）"，不逐页遍历。

### 哈希表原理（感性理解）

存 d["name"] = "小明"：对 "name" 算哈希值（整数指纹），根据哈希值决定存储位置。查 d["name"]：算哈希 -> 直接跳到对应位置取值。

**这就是 O(1) 的奥秘**——不管字典里有 10 个还是 10 万个元素，查找时间不变。

### 哈希表工作原理（进阶理解）

```
1. 计算哈希值: hash("name") -> 某个整数（如 123456789）
2. 转换为索引: 哈希值 % 表大小 -> 槽位索引（如索引 5）
3. 写入/读取: 直接操作槽位 5

哈希冲突处理（开放寻址法）:
    槽位 5 已被占用 -> 探测下一个槽位 6 -> 还被占用 -> 探测 7 -> ...直到找到空位
    查找时也按同样路径探测
```

Python 3.6+ 的字典实现经过重大优化（"紧凑字典"compact dict）：
- 哈希表和数据分开存储，**按插入顺序存储**键值对（Python 3.7+ 语言特性）
- 内存节省约 30-40%（相比旧版 Python）
- 负载因子超过 **2/3** 时自动扩容并 rehash 所有键

### 字典 vs 列表：性能实验

```python
# 在 100 万元素中查找：
big_list = [(i, f"value_{i}") for i in range(1_000_000)]
big_dict = {i: f"value_{i}" for i in range(1_000_000)}

# 列表按值查找：遍历 100 万个元素 -> O(n)
("value_999999" in [v for _, v in big_list])  # 很慢！

# 字典按键查找：一次哈希 -> O(1)
(999999 in big_dict)  # 极快！
```

### 常见字典陷阱

- **遍历时不能修改大小**：`for k in d: d.pop(k)` 会报 RuntimeError
- `d.get(k)` 区分不了"键不存在"和"值就是 None" -> 用 d.get(k, SENTINEL) 配合自定义哨兵
- **键必须可哈希**：list、dict、set 不能做字典键

### Python 3.9+ 字典新特性

```python
d1 = {"a": 1, "b": 2}
d2 = {"b": 3, "c": 4}
merged = d1 | d2          # {"a": 1, "b": 3, "c": 4} -- | 合并运算符
d1 |= d2                  # 原地合并，d1 被更新
```

### 补充：字典推导式

```python
# 键值互换（假设值是唯一的）
original = {"a": 1, "b": 2}
reversed_dict = {v: k for k, v in original.items()}
```

### 补充：.setdefault() 实用方法

```python
# 普通方式：需要判断键是否存在
if "scores" not in student:
    student["scores"] = []
student["scores"].append(95)

# .setdefault() 方式：一行搞定
student.setdefault("scores", []).append(95)
```


In [ ]:
# 字典字面量创建：键值对之间用逗号分隔
# 键（key）必须是**可哈希**的不可变类型（str、int、tuple 等），值（value）可以是任意类型
# 哈希表底层：对每个键调用 hash() 计算哈希值，模除表大小得到槽位索引，O(1) 插入/查找
student = {"name": "小明", "age": 18, "score": 95.5}
print("学生:", student)
# 通过键直接访问 d[key]：计算键的哈希值 → 定位到槽位 → 返回值
# 如果键不存在会抛出 KeyError，这是与 .get() 的主要区别
print("姓名:", student["name"])
# .get(key, default) 是"安全查询"方法：键存在返回值，键不存在返回 default 而不抛异常
# 注意：default 的默认值是 None，所以 .get("grade") 和 .get("grade", None) 等价
grade = student.get("grade", "N/A")  # 安全访问
print("年级:", grade)

# 通过已有的键赋值会覆盖原有值——键的哈希值相同，定位到同一个槽位，替换 Value 指针
student["score"] = 98.0   # 修改
# 赋予一个新键：哈希表在适当位置分配新槽位存储键值对，均摊 O(1)
# 如果负载因子超过 2/3，触发 rehash（重新分配表并重算所有键的哈希），以保持 O(1) 性能
student["city"] = "北京"  # 新增
# del 删除键值对：将对应槽位标记为"已删除"（dummy/tombstone）
# 不能直接清空是因为哈希探测链依赖这些槽位作为跳板，O(1) 均摊
del student["city"]       # 删除
print("修改后:", student)

# 遍历字典
scores = {"小明": 85, "小红": 92, "小刚": 78}
# .items() 返回键值对视图（dict_items），Python 3.7+ 保证按插入顺序遍历
# 每次迭代返回一个 (键, 值) 元组，直接在 for 子句中解包
for name, score in scores.items():
    print(f"  {name}: {score}分")
# .values() 返回值的视图，支持迭代和成员检查（in），O(1) 取长度，O(n) 遍历
print("总分:", sum(scores.values()))
# in 运算符：检查键是否存在，底层为哈希查找，O(1) 平均
print("小明在吗?", "小明" in scores)

# 词频统计——字典最经典的应用模式
text = "apple banana apple orange banana apple"
# .split() 默认按任意空白字符（空格、制表符、换行）分割字符串为单词列表
words = text.split()
count = {}
for word in words:
    # 核心技巧：count.get(word, 0) 在键不存在时返回 0，然后 +1，再赋值回字典
    # 一行代码完成了"初始化并累加"两个操作，避免了手动判断 if word in count:
    count[word] = count.get(word, 0) + 1
print("词频:", count)

# collections.Counter 是 dict 的子类，专为计数场景设计
# 接受任何可迭代对象，遍历并计数，内部实现与上面的手写逻辑完全相同
from collections import Counter
print("Counter:", dict(Counter(words)))
# .most_common(n) 返回出现频率最高的 n 个 (元素, 次数) 对，按频率降序排列
# 底层使用 heapq.nlargest 实现堆排序，时间复杂度 O(n log m)，m 为返回元素数
print("最常见:", Counter(words).most_common(2))

---

## 7. 集合 (Set)——唯一元素的集合

数学集合概念：**不重复 + O(1)成员检查**。

### 为什么集合比列表快？

```
列表查找：遍历每个元素 -> 逐个比较 -> O(n)
集合查找：直接算哈希 -> 定位槽位 -> O(1)

实测对比（1 万元素中查找）：
    list: ~500 微秒（线性扫描）
    set:  ~0.05 微秒（哈希定位）
    差距约 10,000 倍！
```

### 内存模型（仅存键不存值）

```
底层结构：
+-----+-----+-----+-----+-----+-----+-----+-----+
|     |"小" |     |"大" |     |"中" |     |     |  <- 只存元素本身
+-----+-----+-----+-----+-----+-----+-----+-----+
  0     1     2     3     4     5     6     7
                       ^
                  hash("大") % 8 -> 3
```

集合的底层实现与字典几乎相同——用**哈希表**，但每个槽位只存储元素本身（不存储值）：
- 成员检查 O(1) 平均
- 元素必须**可哈希**（实现了 __hash__() 方法）
- 自动去重（重复元素哈希值相同，覆盖旧槽位）

### frozenset：不可变的集合

```python
fs = frozenset([1, 2, 3])
# fs.add(4)  <- frozenset 不可修改
# 但 frozenset 可哈希，可以作为字典键或另一个集合的元素
s = {frozenset([1, 2]), frozenset([3, 4])}  # 集合的集合
```

### 什么元素能放入集合？

需要满足**可哈希**（hashable）：实现了 __hash__() 且 __eq__() 行为正确。

| 可以放入 set | 不能放入 set |
|-------------|-------------|
| int, float, str | list（不可哈希） |
| tuple（内层全为可哈希） | dict（不可哈希） |
| frozenset | set（不可哈希） |
| 自定义对象（需实现 __hash__） | 自定义对象（若 __hash__ 为 None） |

```python
s = {[1, 2, 3]}  # TypeError: unhashable type: 'list'
```

### 集合运算的应用场景

| 运算 | 方法 | 应用场景 |
|------|------|---------|
| a & b | a.intersection(b) | 共同好友、共同爱好 |
| a | b | a.union(b) | 合并去重后的所有数据 |
| a - b | a.difference(b) | 差集：在 A 不在 B |
| a ^ b | a.symmetric_difference(b) | 互斥的权限 |
| a <= b | a.issubset(b) | A 是否 B 的子集 |


In [ ]:
# 花括号创建集合——自动去重，无序（Python 3.7+ 保留插入顺序是 CPython 实现细节，不保证）
# 底层也是哈希表实现，但只存储键（没有值），因此 O(1) 成员检查
fruits = {"苹果", "香蕉", "橘子", "苹果", "香蕉"}
# 输出中重复的"苹果"和"香蕉"只出现一次，因为集合自动去重
print("去重:", fruits)

# 从列表去重：set() 构造函数接受任何可迭代对象
# 遍历列表，将每个元素哈希后写入哈希表，已存在的元素自动覆盖（不报错）
# 注意：去重后顺序不一定与原始列表一致（哈希分布决定槽位顺序）
nums = [1, 2, 2, 3, 3, 3, 4]
print("set去重:", set(nums))

# 常见陷阱：{} 是空字典，不是空集合！
# Python 语法规定：花括号中的内容如果以冒号分隔键值就是字典，否则才是集合
# 空花括号有歧义，被分配给更常见的 dict——空集合必须用 set()
print("空set:", type(set()), "空dict:", type({}))

# 常用方法
s = {1, 2, 3}
# .add() 添加元素，如果元素已存在则无操作（集合不重复），O(1) 均摊
s.add(4)
# .remove() 删除指定元素，元素不存在则抛出 KeyError
# 适合"确定元素存在"的场景，否则需要先 in 检查或使用 discard
s.remove(2)       # 不存在会报错
# .discard() 安全删除：元素存在则删除，不存在则静默无操作
# 相比 remove + try/except，discard 更简洁也更高效（底层直接用 set_discard 操作）
s.discard(99)     # 不存在不报错
print("方法:", s)

# 集合运算——基于哈希表实现，性能远优于列表的逐元素遍历
py = {"小明", "小红", "小刚", "小丽"}
jv = {"小红", "小刚", "小强", "小芳"}
# 并集 |：两个集合中所有不重复的元素，相当于逻辑 OR
# 时间复杂度 O(len(s) + len(t))
print("并集(|):", py | jv)
# 交集 &：同时属于两个集合的元素，遍历较小的集合并检查是否在较大集合中
# 优化策略：Python 自动遍历较小的集合，对较大集合做 O(1) 成员检查
print("交集(&):", py & jv)
# 差集 -：在 py 中但不在 jv 中的元素，注意运算不满足交换律
print("差集(-):", py - jv)
# 对称差 ^：两个集合中只在一个集合中出现的元素（并集 - 交集）
print("对称差(^):", py ^ jv)

---

## 8. 数据结构选择指南

### 决策树

```
需保存多个数据？
+-- 需键值映射？           -> dict
|   +-- 只需判断存在性？    -> set（值不重要时比 dict 省内存）
+-- 不需键值映射 ->
    +-- 需唯一性？          -> set
    +-- 不需 ->
        +-- 需修改？        -> list
        +-- 数据固定？      -> tuple（更省内存、可哈希）
```

### 常见选择场景

| 场景 | 推荐结构 | 原因 |
|------|---------|------|
| 学生成绩表（姓名->分数） | dict | 按键 O(1) 查找分数 |
| IP 地址黑白名单 | set | 只需判断是否存在，O(1) |
| 聊天消息历史 | list | 按时间顺序追加，需保留全部 |
| 函数返回坐标 (x, y) | tuple | 固定两个值，不可变语义 |
| 配置文件常量 | tuple/frozenset | 防止意外修改 |
| 文章单词去重 | set | 自动去重 |

### 性能对比

| 操作 | list | tuple | dict | set |
|------|------|-------|------|-----|
| 按索引/键访问 | O(1) | O(1) | O(1)* | - |
| 按值查找 | O(n) | O(n) | O(1)* | O(1)* |
| 末尾添加 | O(1)** | - | O(1)* | O(1)* |
| 中间插入 | O(n) | - | - | - |
| 删除 | O(n) | - | O(1)* | O(1)* |
| 成员检查 (in) | O(n) | O(n) | O(1)* | O(1)* |
| 内存开销 | 较少 | **最少** | 较多 | 较多 |

> *O(1)=平均常数时间，O(n)=线性时间，**=均摊 O(1)

### 常见选择错误

1. **"用 list 做成员检查"**：if x in big_list 是 O(n)，改用 set
2. **"用 list 当 dict 键"**：必须用 tuple（列表不可哈希）
3. **"不在乎内存用 dict"**：如果只需要成员检查，set 占用内存大约是 dict 的一半
4. **"tuple 只是不能改的 list"**：tuple 还有可哈希、更省内存、语义明确等优势

### 内存开销量化（近似值，64 位 Python）

| 结构 | 每个元素额外开销 | 说明 |
|------|----------------|------|
| list | 约 8 字节（指针）+ 预留容量 | 空列表约 56 字节 |
| tuple | 0 额外开销 | 空元组约 40 字节（单块分配） |
| dict | 约 30-50 字节/键值对 | 哈希表本身 + 探测链 |
| set | 约 20-30 字节/元素 | 和 dict 类似但省去了值数组 |


In [ ]:
# 同一份数据用四种数据结构展示，体会不同视角下的差异
data = [3, 1, 2, 3, 1, 4, 5]

# list：保留所有元素，包括顺序、重复值和插入顺序——适合需要索引访问（O(1)）或保留完整数据的场景
print("list(保留所有):", data)
# tuple：将列表转为不可变序列（浅层不可变），适合作为字典键或防止意外修改
# 注意：如果列表元素中包含可变对象（如嵌套列表），tuple 只是禁止修改顶层引用，不能冻结内层
print("tuple(只读):", tuple(data))
# set：自动去重，但丢失顺序信息。sorted() 恢复排序后得到有序的去重结果
# 去重原理：哈希表逐个插入，重复元素覆盖（O(n) 完成去重）
print("set(去重):", sorted(set(data)))

# 手动统计频次：字典是实现"计数"最自然的数据结构
# 对于每个元素，用 get() 实现"不存在则初始化为 0 再 +1"
freq = {}
for x in data:
    freq[x] = freq.get(x, 0) + 1
# 最终每个数字出现几次一目了然——字典把"一一对应"的关系表达得最清晰
print("dict(频次):", freq)

---

## 9. 综合练习：联系人通讯录

综合运用 dict（主数据，O(1) 姓名查找）、list（sorted 排序）、set（去重检测）。

### 为什么选择 dict 作为核心存储？

通讯录的核心操作是"按姓名查找电话号码"——这正是 dict 最擅长的场景：
- contacts[name] = phone -> O(1) 写入
- contacts.get(name)     -> O(1) 读出
- name in contacts       -> O(1) 存在性检查

如果用 list 存储 (name, phone) 元组列表，按姓名查找需要 O(n) 遍历——1000 个联系人时 dict 快 1000 倍。

### 当前实现的局限与改进方向

- **重复姓名**：当前策略是覆盖（更新），也可改为存储列表（一人多号）
- **排序输出**：用 sorted(contacts) 按姓名排序，O(n log n)
- **模糊查找**：可以用 `[name for name in contacts if keyword in name]` 实现名称搜索


In [ ]:
# 通讯录应用——综合运用 dict（核心存储）、list（sorted 排序）、set（可选去重检测）
contacts = {}  # name -> phone

def add(name, phone):
    # 直接赋值给字典：如果 name 已存在则覆盖旧号码（更新），不存在则新增
    # 由于键 name 是字符串（可哈希类型），底层 O(1) 写入哈希表
    contacts[name] = phone
    # f-string 格式化字符串，Python 3.6+ 引入，比 % 格式化和 .format() 更简洁高效
    print(f"已添加: {name} -> {phone}")

def delete(name):
    # in 运算符检查键是否存在——字典的成员检查时间复杂度为 O(1)（哈希查找）
    if name in contacts:
        # del 删除键值对，底层将槽位标记为 tombstone（墓碑标记），O(1) 均摊
        del contacts[name]
        print(f"已删除: {name}")
    else:
        # 如果不检查直接 del 不存在的键会抛出 KeyError，所以必须预先检查或使用 try/except
        print(f"'{name}' 不存在")

def find(name):
    # .get() 安全查询：键存在返回值，不存在返回 None（默认值），不抛异常
    # 潜在问题：如果电话号码存储为 None，.get() 无法区分"存在但为 None"和"不存在"
    phone = contacts.get(name)
    # 三元条件表达式（条件为真时取 if 前，否则取 else 后）
    # 注意：if phone 会将 phone=None 时视为假值，此时输出"未找到"
    # 如果电话号码可能为 0 或空字符串等假值，需用 if phone is not None 更严谨
    print(f"{name}: {phone}" if phone else f"未找到: {name}")

def list_all():
    if not contacts:
        print("通讯录为空"); return
    print("\n======= 通讯录 =======")
    # sorted() 对字典键排序，返回排序后的键列表，Timsort 算法 O(n log n)
    # .keys() 可省略（直接 sorted(contacts) 等价），但显式写出增强可读性
    for name in sorted(contacts.keys()):
        # 每次循环做一次 O(1) 字典查找获取对应电话号码
        print(f"  {name}: {contacts[name]}")
    print(f"共 {len(contacts)} 人")

# 演示
add("张三", "13800138001")
add("李四", "13800138002")
add("王五", "13800138003")
# 添加已存在名字——字典无重复键，旧号码被新号码覆盖（适合更新场景）
add("张三", "13900990001")  # 覆盖旧号码
list_all()
find("李四")
delete("王五")
list_all()

---

## 总结与练习

### 口诀

> 列表有序可修改，元组锁定不变更。字典键值查得快，集合去重挑唯一。

### 四种结构一句话记忆

| 结构 | 一句话 | 时间复杂度核心要点 |
|------|--------|-------------------|
| list | 动态数组，索引 O(1)，查找 O(n) | 索引快，成员检查慢 |
| tuple | 只读列表，省内存，可哈希 | 固定数据首选 |
| dict | 键值映射，哈希 O(1) 查找 | 快速查询，键需可哈希 |
| set | 唯一集合，哈希 O(1) 成员检查 | 去重和关系运算 |

### 常见陷阱快速回顾

- **复制陷阱**：b = a 不复制，用 a.copy() 或 a[:] 做浅拷贝
- **默认参数**：不要用可变对象（list/dict/set）做函数默认参数
- **空集合**：{} 是空字典，空集合必须 set()
- **遍历时修改**：遍历列表/字典时不要增删元素
- **哈希要求**：dict 的键和 set 的元素必须是可哈希的不可变类型

### 课后练习

1. **列表推导**：生成 1-50 中能被 3 整除的数的平方，取前 5 个（一行）。
2. **字典查询**：城市->人口字典，用户输入查人口，不存在给提示。
3. **集合交集**：两个班级名单，找出都报名的（模拟面试，不用 & 运算）。
4. **词频分析**：统计给定文本中出现最多的 3 个词。

### 进一步学习

掌握这些基础数据结构后，可以进一步学习：
- collections 模块：deque、defaultdict、Counter、OrderedDict
- heapq 堆队列：优先级队列
- array 模块：更省内存的同质数组
- 自定义类实现 __getitem__、__len__ 模拟序列行为
